# 18_descriptors_for_weka — 1:1 학습셋 descriptor 계산 & WEKA용 CSV 준비

**이 노트북이 하는 일:** 1:1 학습셋(active + 실측 inactive + decoy)에 들어있는 각 분자에 대해
RDKit 2D **descriptor(물리화학·위상 수치) 217종**을 계산하고, 두 종류의 파일을 만든다.

1. **전체 Excel** — 분자 정보(SMILES 등) + potency + descriptor 217개 (사람이 보고 확인용)
2. **WEKA용 CSV** — WEKA(feature selection 도구)에 바로 넣을 수 있게 **숫자형 descriptor만 + 클래스(potency)를 마지막 열**로, 문자열·결측 없이 정제

**왜 필요한가:** 217개 descriptor에는 서로 겹치는(상관 높은) 것이 많아서, 학습 전에 WEKA로
**정말 유용한 것만 골라내기(feature selection)** 위한 입력 파일을 준비하는 단계다.

### 셀 1 — 준비(폴더 이동 + 라이브러리)
어느 폴더에서 노트북을 열어도 프로젝트 최상위에서 실행되도록 이동하고, 필요한 도구를 불러온다.
`Descriptors`가 RDKit의 descriptor 계산 모듈이다.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')                     # data 폴더가 안 보이면 상위로 한 칸 이동
print('작업 폴더:', os.getcwd())

import numpy as np                      # 수치 계산
import pandas as pd                     # 표(DataFrame) 처리
from rdkit import Chem                  # SMILES → 분자 객체
from rdkit.Chem import Descriptors      # 분자 descriptor 계산 함수 모음
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')          # RDKit 경고 로그 끄기(대량 처리 시 노이즈 방지)

### 셀 2 — 1:1 학습셋 불러오기
`train_1to1.csv`에는 canonical_smiles, inchikey, label(1/0), source(active/real_inactive/decoy)가 있다.
학습 클래스 이름을 명확히 하려고 `label`을 **`potency`** 로 바꾼다(1=active, 0=inactive).
분포를 출력해 active와 inactive가 2049개씩 균형인지 확인한다.

In [ ]:
# 1:1 학습셋 로드 (active + 실측 inactive + decoy). label(1/0) -> potency
SRC = 'data/train_1to1.csv'
df = pd.read_csv(SRC)
df = df.rename(columns={'label': 'potency'})            # 클래스 열 이름을 potency로
print('[0] 1:1 학습셋 shape:', df.shape)
print('    potency 분포:', dict(df.potency.value_counts()))   # 1/0 개수
print('    source 분포:', dict(df.source.value_counts()))     # active/decoy/real_inactive 개수

### 셀 3 — descriptor 217종 계산 → 전체 Excel 저장
`Descriptors._descList`는 RDKit에 등록된 모든 descriptor의 (이름, 함수) 목록이다.
각 분자마다 `CalcMolDescriptors`로 217개 값을 한 번에 구한다. 분자 파싱이 실패하면 건너뛴다.
계산 결과(X)에 분자 메타정보(meta)를 붙여 하나의 표(full)로 만들고 Excel로 저장한다.
(217종 × 수천 개라 수 분 걸린다.)

In [ ]:
# RDKit 2D descriptor 217종 계산 -> 전체 Excel 저장(메타 + potency + descriptor)
desc_names = [n for n, _ in Descriptors._descList]      # 등록된 descriptor 이름 217개
print('descriptor', len(desc_names), '종 계산 중... (수 분 소요)')

rows, keep = [], []                                     # rows=descriptor값들, keep=성공한 행 인덱스
for i, smi in enumerate(df['canonical_smiles']):
    m = Chem.MolFromSmiles(str(smi))                    # SMILES → 분자
    if m is None:                                       # 파싱 실패하면 건너뜀
        continue
    d = Descriptors.CalcMolDescriptors(m)               # descriptor 217개를 dict로 계산
    rows.append([d.get(n, np.nan) for n in desc_names]) # 이름 순서대로 값 추출
    keep.append(i)
    if (i + 1) % 1000 == 0:
        print('  ', i + 1, '/', len(df))                # 진행 상황 표시

X = pd.DataFrame(rows, columns=desc_names)              # descriptor 표
meta = df.iloc[keep][['canonical_smiles', 'inchikey', 'source', 'potency']].reset_index(drop=True)
full = pd.concat([meta, X.reset_index(drop=True)], axis=1)   # 메타 + descriptor 결합

XLSX = 'data/HSD17B13_1to1_descriptors.xlsx'
full.to_excel(XLSX, index=False)
print('[Excel] 전체 저장:', XLSX, '| shape', full.shape,
      '(메타 4 + descriptor', len(desc_names), ')')

### 셀 4 — WEKA용 CSV로 정제
WEKA feature selection에 넣으려면 **① 숫자형 descriptor만, ② 클래스(potency)는 마지막 열, ③ 문자열 컬럼 제거, ④ 결측값 없음** 이어야 한다.
- `remove_invalid_descriptors`: 숫자로 변환이 안 되는(문자/이상값) 열을 찾아 제거
- `inf → NaN` 변환 후, 결측이 든 **행**을 제거해 완전한 수치 행렬로 만듦(모든 descriptor 열은 유지)
- 마지막에 `potency`를 **맨 끝 열**로 붙여 CSV 저장 (실제 feature 선택은 WEKA에서 수행)

In [ ]:
# ===== WEKA용 CSV 정제 =====
# 규칙: 숫자형 descriptor만 + 클래스(potency)는 '마지막 열' + 문자열 컬럼 삭제 + 결측 없음
compound_info = full[['canonical_smiles', 'inchikey', 'source', 'potency']]  # 메타(문자열 등)
descriptor_data = full[desc_names]                                            # descriptor만 추출
print('[1] descriptor 데이터 shape:', descriptor_data.shape)

# 1) 숫자로 변환 안 되는(문자/이상값) 컬럼 제거
def remove_invalid_descriptors(data):
    invalid = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col], errors='raise')    # 숫자 변환 시도, 실패하면 예외
        except Exception:
            invalid.append(col)                         # 변환 실패한 열 = 문자/이상값
    print('[2] 숫자변환 실패(문자/이상) 컬럼 제거:', len(invalid), '개')
    return data.drop(columns=invalid)

desc_num = remove_invalid_descriptors(descriptor_data).apply(pd.to_numeric)   # 전부 숫자형으로

# 2) inf -> NaN 후 결측 처리: NaN 있는 '행' 제거(모든 descriptor 컬럼은 유지)
desc_num = desc_num.replace([np.inf, -np.inf], np.nan)  # 무한대를 결측으로
n_nan_cell = int(desc_num.isna().sum().sum())           # 결측 셀 개수
nan_row_mask = desc_num.isna().any(axis=1)              # 결측이 하나라도 있는 행
print('[3] 결측/inf 셀', n_nan_cell, '개 -> 결측 포함 행', int(nan_row_mask.sum()), '개 제거')
desc_num = desc_num[~nan_row_mask].reset_index(drop=True)
potency = compound_info['potency'][~nan_row_mask.values].reset_index(drop=True)

# 3) 클래스(potency)를 '마지막 열'에 두고, 문자열 컬럼은 모두 제외
weka_df = desc_num.copy()
weka_df['potency'] = potency.values                     # potency를 맨 끝에 추가
print('[4] WEKA용 shape (descriptor + potency):', weka_df.shape)
print('    마지막 열:', weka_df.columns[-1], '| 클래스 분포:', dict(weka_df.potency.value_counts()))

CSV = 'data/HSD17B13_1to1_descriptors_weka.csv'
weka_df.to_csv(CSV, index=False)
print('[5] WEKA용 CSV 저장 완료:', CSV)
print('    (WEKA Explorer -> Open file -> Select attributes 탭에서 사용)')